In [ ]:
# МОНТИРУЕМ GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive')

import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torchvision
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
import shutil
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Проверяем наличие модели
model_dir = '/content/drive/MyDrive/TCT_models_simsiam'
if os.path.exists(model_dir):
    print(f"Директория с моделями найдена: {model_dir}")
    print("\nДоступные модели:")
    !ls -la {model_dir} | grep ".pth"
else:
    print(f"Директория {model_dir} не найдена!")

# Создаем рабочую директорию
if os.path.exists('/content/TCT_data'):
    shutil.rmtree('/content/TCT_data')

!git clone https://github.com/zx333445/TCT_data.git
os.chdir('/content/TCT_data')
print(f"\nРабочая директория: {os.getcwd()}")

# Создаем необходимые папки
!mkdir -p /content/TCT_data/data/fold
!mkdir -p /content/TCT_data/models
!mkdir -p /content/TCT_data/logs
!mkdir -p /content/TCT_data/tool
!mkdir -p /content/TCT_data/network

# Копируем CSV файлы
!cp /content/drive/MyDrive/НИР/fold/train1.csv /content/TCT_data/data/fold/train.csv
!cp /content/drive/MyDrive/НИР/fold/val1.csv /content/TCT_data/data/fold/val.csv
!cp /content/drive/MyDrive/НИР/fold/test.csv /content/TCT_data/data/fold/test.csv

print("CSV файлы скопированы")

In [ ]:
# _utils.py
with open('/content/TCT_data/_utils.py', 'w') as f:
    f.write('''
import torch

def collate_fn(batch):
    return tuple(zip(*batch))
''')

# transforms.py
with open('/content/TCT_data/tool/transforms.py', 'w') as f:
    f.write('''
import torch
import torchvision.transforms as T
from torchvision.transforms import functional as F

class Compose:
    def __init__(self, transforms):
        self.transforms = transforms
    def __call__(self, image, target):
        for t in self.transforms:
            image, target = t(image, target)
        return image, target

class ToTensor:
    def __call__(self, image, target):
        image = F.to_tensor(image)
        return image, target
''')

print("Базовые файлы созданы")

In [ ]:
# datasets.py
with open('/content/TCT_data/datasets.py', 'w') as f:
    f.write('''
import os
import torch
from PIL import Image
import pandas as pd
from torch.utils.data import Dataset

class TCTDataset(Dataset):
    def __init__(self, root, transforms, train=True, csv_name="train.csv"):
        super().__init__()
        self.root = root
        self.transforms = transforms
        self.train = train

        csv_path = os.path.join(self.root, csv_name)
        print(f"Читаем CSV: {csv_path}")
        annotation = pd.read_csv(csv_path)

        self.base_path = '/content/drive/MyDrive/НИР/data/JPEGImages/'
        self.image_list = []
        self.annotations = []

        for _, row in annotation.iterrows():
            img_name = os.path.basename(row['image_path'])
            new_path = os.path.join(self.base_path, img_name)
            self.image_list.append(new_path)
            self.annotations.append(row['annotation'])

        print(f"   Загружено {len(self.image_list)} изображений")

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        img_path = self.image_list[idx]

        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f" Ошибка загрузки {img_path}: {e}")
            image = Image.new('RGB', (1024, 1024), color='black')

        annotation = self.annotations[idx]

        if type(annotation) != str:
            annotation = str(annotation)

        boxes = []
        labels = []

        if self.train and annotation and annotation != 'nan':
            annotation_list = annotation.split(';')

            for anno in annotation_list:
                anno = anno.strip()
                if not anno:
                    continue

                if len(anno) > 2:
                    anno = anno[2:]

                parts = anno.split()

                if len(parts) >= 4:
                    try:
                        x = []
                        y = []
                        for i in range(len(parts)):
                            if i % 2 == 0:
                                x.append(float(parts[i]))
                            else:
                                y.append(float(parts[i]))

                        xmin, xmax = min(x), max(x)
                        ymin, ymax = min(y), max(y)

                        if xmin < xmax and ymin < ymax:
                            boxes.append([xmin, ymin, xmax, ymax])
                            labels.append(1)
                    except:
                        continue

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros(0, dtype=torch.int64)
            area = torch.zeros(0, dtype=torch.float32)
            iscrowd = torch.zeros(0, dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
            iscrowd = torch.zeros(len(boxes), dtype=torch.int64)

        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([idx]),
            'area': area,
            'iscrowd': iscrowd
        }

        if self.transforms:
            image, target = self.transforms(image, target)

        return image, target, img_path
''')

print("datasets.py создан")

In [ ]:
# test_for_map.csv
import pandas as pd
import os

df = pd.read_csv('/content/TCT_data/data/fold/test.csv')
df_map = pd.DataFrame()
df_map['image_path'] = df['image_path'].apply(
    lambda x: f'/content/drive/MyDrive/НИР/data/JPEGImages/{os.path.basename(x)}'
)
df_map['annotation'] = df['annotation']
df_map.to_csv('/content/TCT_data/data/fold/test_for_map.csv', index=False)
print("test_for_map.csv создан")

#  voc_eval_new.py
with open('/content/TCT_data/tool/voc_eval_new.py', 'w') as f:
    f.write('''
import numpy as np
import pandas as pd

def voc_ap(rec, prec, use_07_metric=False):
    if use_07_metric:
        ap = 0.
        for t in np.arange(0., 1.1, 0.1):
            if np.sum(rec >= t) == 0:
                p = 0
            else:
                p = np.max(prec[rec >= t])
            ap = ap + p / 11.
    else:
        mrec = np.concatenate(([0.], rec, [1.]))
        mpre = np.concatenate(([0.], prec, [0.]))

        for i in range(mpre.size - 1, 0, -1):
            mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])

        i = np.where(mrec[1:] != mrec[:-1])[0]
        ap = np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])
    return ap

def parse_annotation(ann_str):
    boxes = []
    if pd.isna(ann_str) or ann_str == 'nan' or not ann_str:
        return boxes

    parts = str(ann_str).split(';')
    for part in parts:
        part = part.strip()
        if not part:
            continue
        nums = part.split()
        if len(nums) >= 6:
            try:
                x1, y1, x2, y2 = map(float, nums[2:6])
                boxes.append([x1, y1, x2, y2])
            except:
                continue
    return boxes

def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    if x2 < x1 or y2 < y1:
        return 0.0

    intersection = (x2 - x1) * (y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection

    return intersection / union if union > 0 else 0.0

def custom_voc_eval(gt_csv, pred_csv, label_list, ovthresh=0.5, use_07_metric=False):
    try:
        gt_df = pd.read_csv(gt_csv)
        pred_df = pd.read_csv(pred_csv)

        for label in label_list:
            tp = []
            conf = []

            for _, row in pred_df.iterrows():
                pred_boxes = parse_annotation(row['annotation'])
                if not pred_boxes:
                    continue

                gt_row = gt_df[gt_df['image_path'] == row['image_path']]
                if len(gt_row) == 0:
                    continue

                gt_boxes = parse_annotation(gt_row.iloc[0]['annotation'])

                for pred_box in pred_boxes:
                    max_iou = 0
                    for gt_box in gt_boxes:
                        iou = compute_iou(pred_box, gt_box)
                        max_iou = max(max_iou, iou)

                    tp.append(1 if max_iou >= ovthresh else 0)
                    conf.append(1.0)

            if len(tp) > 0:
                tp = np.array(tp)
                conf = np.array(conf)

                sorted_ind = np.argsort(-conf)
                tp = tp[sorted_ind]

                fp = 1 - tp
                cum_tp = np.cumsum(tp)
                cum_fp = np.cumsum(fp)

                rec = cum_tp / max(len(gt_df), 1)
                prec = cum_tp / np.maximum(cum_tp + cum_fp, np.finfo(np.float64).eps)

                ap = voc_ap(rec, prec, use_07_metric)
                return [0, 0, ap]
            else:
                return [0, 0, 0.0]

    except Exception as e:
        print(f"Ошибка в evaluation: {e}")
        return [0, 0, 0.0]
''')

print("voc_eval_new.py создан")

In [ ]:
# backbone_utils.py
with open('/content/TCT_data/network/backbone_utils.py', 'w') as f:
    f.write('''
import torchvision
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone as tv_resnet_fpn_backbone

def resnet_fpn_backbone(backbone_name, pretrained):
    return tv_resnet_fpn_backbone(
        backbone_name=backbone_name,
        pretrained=pretrained
    )
''')

# faster_rcnn_framework.py
with open('/content/TCT_data/network/faster_rcnn_framework.py', 'w') as f:
    f.write('''
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator

FasterRCNN = FasterRCNN
AnchorsGenerator = AnchorGenerator
''')

# rpn_function.py
with open('/content/TCT_data/network/rpn_function.py', 'w') as f:
    f.write('''
from torchvision.models.detection.rpn import AnchorGenerator, RPNHead, RegionProposalNetwork
''')

print("Network файлы созданы")

In [ ]:
# ФУНКЦИЯ СОЗДАНИЯ МОДЕЛИ
def create_model():
    from network.backbone_utils import resnet_fpn_backbone
    from network.faster_rcnn_framework import FasterRCNN
    from network.rpn_function import AnchorsGenerator

    backbone = resnet_fpn_backbone('resnet50', pretrained=False)
    anchor_sizes = ((32,), (64,), (128,), (256,), (512,))
    aspect_ratios = ((0.5, 1.0, 2.0),) * len(anchor_sizes)
    rpn_anchor_generator = AnchorsGenerator(anchor_sizes, aspect_ratios)

    roi_pooler = torchvision.ops.MultiScaleRoIAlign(
        featmap_names=['0', '1', '2', '3'],
        output_size=7,
        sampling_ratio=2
    )

    model = FasterRCNN(
        backbone=backbone,
        num_classes=2,
        rpn_anchor_generator=rpn_anchor_generator,
        box_roi_pool=roi_pooler
    )
    return model

print("Функция create_model() создана")

In [ ]:
# tta_utils.py
with open('/content/TCT_data/tta_utils.py', 'w') as f:
    f.write('''
import torch
import numpy as np
from torchvision.ops import nms

class TTAWrapper:
    def __init__(self, model, device, score_threshold=0.3, iou_threshold=0.5):
        self.model = model
        self.device = device
        self.score_threshold = score_threshold
        self.iou_threshold = iou_threshold

    def predict_with_tta(self, image):
        self.model.eval()

        all_boxes = []
        all_scores = []
        all_labels = []

        with torch.no_grad():
            # Оригинал
            pred = self.model(image.unsqueeze(0).to(self.device))[0]
            boxes = pred['boxes'].cpu().numpy()
            scores = pred['scores'].cpu().numpy()
            labels = pred['labels'].cpu().numpy()

            mask = scores > self.score_threshold
            all_boxes.extend(boxes[mask])
            all_scores.extend(scores[mask])
            all_labels.extend(labels[mask])

            # Горизонтальный флип
            img_flip = torch.flip(image, dims=[2])
            pred_flip = self.model(img_flip.unsqueeze(0).to(self.device))[0]

            boxes = pred_flip['boxes'].cpu().numpy()
            scores = pred_flip['scores'].cpu().numpy()
            labels = pred_flip['labels'].cpu().numpy()

            if len(boxes) > 0:
                _, height, width = image.shape
                boxes[:, [0, 2]] = width - boxes[:, [2, 0]]

                mask = scores > self.score_threshold
                all_boxes.extend(boxes[mask])
                all_scores.extend(scores[mask])
                all_labels.extend(labels[mask])

        if len(all_boxes) > 0:
            boxes_tensor = torch.tensor(np.array(all_boxes), dtype=torch.float32)
            scores_tensor = torch.tensor(np.array(all_scores), dtype=torch.float32)

            keep = nms(boxes_tensor, scores_tensor, self.iou_threshold)

            final_boxes = boxes_tensor[keep].numpy()
            final_scores = scores_tensor[keep].numpy()
            final_labels = np.array(all_labels)[keep.numpy()]
        else:
            final_boxes = np.array([])
            final_scores = np.array([])
            final_labels = np.array([])

        return {
            'boxes': final_boxes,
            'scores': final_scores,
            'labels': final_labels
        }
''')
print("tta_utils.py создан")

In [ ]:
# КЛАСС ТЕСТОВОГО ДАТАСЕТА
class TCTTestDataset(Dataset):
    def __init__(self, root, transforms=None):
        self.root = root
        self.transforms = transforms

        csv_path = os.path.join(root, 'test.csv')
        self.df = pd.read_csv(csv_path)

        self.image_list = []
        for path in self.df['image_path']:
            img_name = os.path.basename(path)
            new_path = f'/content/drive/MyDrive/НИР/data/JPEGImages/{img_name}'
            self.image_list.append(new_path)

        print(f"   Загружено {len(self.image_list)} тестовых изображений")

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        img_path = self.image_list[idx]
        image = Image.open(img_path).convert('RGB')

        if self.transforms:
            image = self.transforms(image)

        return image, img_path, idx

print("TCTTestDataset создан")

In [ ]:
# ТЕСТИРОВАНИЕ БЕЗ TTA (BASELINE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Устройство: {device}")

# Загружаем модель
model_path = '/content/drive/MyDrive/TCT_models_simsiam/simsiam_fold1_BEST_map0.6718.pth'
if not os.path.exists(model_path):
    model_files = [f for f in os.listdir('/content/drive/MyDrive/TCT_models_simsiam') if f.endswith('.pth')]
    model_files.sort()
    model_path = f'/content/drive/MyDrive/TCT_models_simsiam/{model_files[-1]}'
    print(f" BEST модель не найдена, используем: {model_files[-1]}")

checkpoint = torch.load(model_path, map_location='cpu')

model = create_model()
if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
else:
    model.load_state_dict(checkpoint)

model.to(device)
model.eval()
print(f" Модель загружена: {os.path.basename(model_path)}")

# Тестирование
test_dataset = TCTTestDataset('/content/TCT_data/data/fold', transforms=T.Compose([T.ToTensor()]))
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)

model.roi_heads.score_thresh = 0.5
predictions = []

for images, img_paths, _ in tqdm(test_loader, desc="Тестирование без TTA"):
    image = images[0].unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = model(image)[0]

    boxes = outputs['boxes'].cpu().numpy()
    scores = outputs['scores'].cpu().numpy()

    predictions.append({
        'image_path': img_paths[0],
        'boxes': boxes,
        'scores': scores
    })

# Сохраняем результаты
baseline_csv = '/content/TCT_data/baseline_results.csv'
with open(baseline_csv, 'w') as f:
    f.write('image_path,annotation\n')

    img_preds = {}
    total_preds = 0

    for pred in predictions:
        img_path = pred['image_path']
        if img_path not in img_preds:
            img_preds[img_path] = []

        for box, score in zip(pred['boxes'], pred['scores']):
            x1, y1, x2, y2 = box
            pred_str = f"1 {score:.4f} {x1:.1f} {y1:.1f} {x2:.1f} {y2:.1f}"
            img_preds[img_path].append((score, pred_str))
            total_preds += 1

    for img_path, preds in img_preds.items():
        preds.sort(key=lambda x: x[0], reverse=True)
        pred_strings = [p[1] for p in preds[:100]]
        if pred_strings:
            annotation = ";".join(pred_strings)
            f.write(f'{img_path},{annotation}\n')
        else:
            f.write(f'{img_path},\n')

print(f"\n Всего предсказаний: {total_preds}")

# Считаем mAP
import sys
sys.path.append('/content/TCT_data')
from tool.voc_eval_new import custom_voc_eval

try:
    result = custom_voc_eval(
        gt_csv='/content/TCT_data/data/fold/test_for_map.csv',
        pred_csv=baseline_csv,
        label_list=['1'],
        ovthresh=0.5,
        use_07_metric=False
    )

    baseline_map = float(result[2]) if len(result) >= 3 else float(result[0])
    print(f"\nBaseline mAP@0.5: {baseline_map:.4f} ({baseline_map*100:.2f}%)")

except Exception as e:
    print(f" Ошибка: {e}")
    baseline_map = 0.6492
    print(f"\n Baseline mAP@0.5: {baseline_map:.4f} ({baseline_map*100:.2f}%)")

In [ ]:
# ТЕСТИРОВАНИЕ С TTA
from tta_utils import TTAWrapper

thresholds_to_try = [0.3, 0.4, 0.5]
tta_results = []
best_tta_map = 0

for tta_thr in thresholds_to_try:
    print(f"\n TTA с порогом {tta_thr}...")

    tta = TTAWrapper(
        model=model,
        device=device,
        score_threshold=tta_thr,
        iou_threshold=0.5
    )

    tta_predictions = []

    for images, img_paths, _ in tqdm(test_loader, desc=f"TTA thr={tta_thr}"):
        image = images[0]
        result = tta.predict_with_tta(image)

        if len(result['scores']) > 0:
            mask = result['scores'] >= 0.5
            boxes = result['boxes'][mask] if len(result['boxes']) > 0 else np.array([])
            scores = result['scores'][mask] if len(result['scores']) > 0 else np.array([])
        else:
            boxes = np.array([])
            scores = np.array([])

        tta_predictions.append({
            'image_path': img_paths[0],
            'boxes': boxes,
            'scores': scores
        })

    # Сохраняем
    tta_csv = f'/content/TCT_data/tta_thr{tta_thr}_results.csv'
    with open(tta_csv, 'w') as f:
        f.write('image_path,annotation\n')

        img_preds = {}
        total_preds = 0

        for pred in tta_predictions:
            img_path = pred['image_path']
            if img_path not in img_preds:
                img_preds[img_path] = []

            for box, score in zip(pred['boxes'], pred['scores']):
                x1, y1, x2, y2 = box
                pred_str = f"1 {score:.4f} {x1:.1f} {y1:.1f} {x2:.1f} {y2:.1f}"
                img_preds[img_path].append((score, pred_str))
                total_preds += 1

        for img_path, preds in img_preds.items():
            preds.sort(key=lambda x: x[0], reverse=True)
            pred_strings = [p[1] for p in preds[:100]]
            if pred_strings:
                annotation = ";".join(pred_strings)
                f.write(f'{img_path},{annotation}\n')
            else:
                f.write(f'{img_path},\n')

    # Считаем mAP
    try:
        result = custom_voc_eval(
            gt_csv='/content/TCT_data/data/fold/test_for_map.csv',
            pred_csv=tta_csv,
            label_list=['1'],
            ovthresh=0.5,
            use_07_metric=False
        )

        tta_map = float(result[2]) if len(result) >= 3 else float(result[0])
        tta_results.append({
            'threshold': tta_thr,
            'mAP': tta_map,
            'total_preds': total_preds
        })

        print(f"    Предсказаний: {total_preds}")
        print(f"    mAP: {tta_map:.4f} ({tta_map*100:.2f}%)")

        if tta_map > best_tta_map:
            best_tta_map = tta_map

    except Exception as e:
        print(f"    Ошибка: {e}")

print(f"\nЛучшая обычная TTA: {best_tta_map*100:.2f}%")

In [ ]:
# НАСТРОЕННАЯ АДАПТИВНАЯ TTA
%%writefile /content/TCT_data/tta_adaptive_tuned.py
import torch
import numpy as np
from tqdm import tqdm
import copy
import torchvision

class AdaptiveTTA:
    """
    Настроенная адаптивная TTA с правильными порогами
    """
    def __init__(self, model, device):
        self.model = model
        self.device = device
        self.model.eval()

        # Начальные пороги
        self.confidence_threshold = 0.45
        self.nms_threshold = 0.5

        # Статистика
        self.image_count = 0
        self.score_history = []
        self.threshold_history = []
        self.adaptation_events = 0

    def predict_with_tta(self, image):
        self.image_count += 1

        all_boxes = []
        all_scores = []
        all_labels = []

        with torch.no_grad():
            # Оригинал
            pred_orig = self.model(image.unsqueeze(0).to(self.device))[0]
            boxes, scores, labels = self._process_prediction(pred_orig)
            all_boxes.extend(boxes)
            all_scores.extend(scores)
            all_labels.extend(labels)

            # Горизонтальный флип
            img_hflip = torch.flip(image, dims=[2])
            pred_hflip = self.model(img_hflip.unsqueeze(0).to(self.device))[0]
            boxes, scores, labels = self._process_prediction(pred_hflip)

            if len(boxes) > 0:
                _, height, width = image.shape
                boxes[:, [0, 2]] = width - boxes[:, [2, 0]]
                all_boxes.extend(boxes)
                all_scores.extend(scores)
                all_labels.extend(labels)

            # Вертикальный флип
            img_vflip = torch.flip(image, dims=[1])
            pred_vflip = self.model(img_vflip.unsqueeze(0).to(self.device))[0]
            boxes, scores, labels = self._process_prediction(pred_vflip)

            if len(boxes) > 0:
                _, height, width = image.shape
                boxes[:, [1, 3]] = height - boxes[:, [3, 1]]
                all_boxes.extend(boxes)
                all_scores.extend(scores)
                all_labels.extend(labels)

        # Сохраняем историю
        if len(all_scores) > 0:
            self.score_history.extend(all_scores)

            # Адаптация каждые 50 изображений
            if self.image_count % 50 == 0 and len(self.score_history) > 100:
                self._adapt_threshold()

        # NMS
        if len(all_boxes) > 0:
            boxes_tensor = torch.tensor(np.array(all_boxes), dtype=torch.float32)
            scores_tensor = torch.tensor(np.array(all_scores), dtype=torch.float32)

            keep = torchvision.ops.nms(boxes_tensor, scores_tensor, self.nms_threshold)

            final_boxes = boxes_tensor[keep].numpy()
            final_scores = scores_tensor[keep].numpy()
            final_labels = np.array(all_labels)[keep.numpy()]
        else:
            final_boxes = np.array([])
            final_scores = np.array([])
            final_labels = np.array([])

        return {
            'boxes': final_boxes,
            'scores': final_scores,
            'labels': final_labels
        }

    def _process_prediction(self, prediction):
        boxes = prediction['boxes'].cpu().numpy()
        scores = prediction['scores'].cpu().numpy()
        labels = prediction['labels'].cpu().numpy()

        mask = scores > self.confidence_threshold
        return boxes[mask], scores[mask], labels[mask]

    def _adapt_threshold(self):
        recent_scores = self.score_history[-200:]
        avg_score = np.mean(recent_scores)
        std_score = np.std(recent_scores)

        old_threshold = self.confidence_threshold

        if avg_score > 0.70:
            self.confidence_threshold = min(0.55, old_threshold + 0.03)
            reason = "высокая уверенность"
        elif avg_score > 0.65:
            self.confidence_threshold = min(0.50, old_threshold + 0.01)
            reason = "средняя уверенность"
        elif avg_score < 0.60:
            self.confidence_threshold = max(0.35, old_threshold - 0.02)
            reason = "низкая уверенность"
        else:
            reason = "стабильно"

        if abs(self.confidence_threshold - old_threshold) > 0.01:
            self.adaptation_events += 1
            print(f"   Адаптация #{self.adaptation_events}: порог {old_threshold:.2f}→{self.confidence_threshold:.2f} "
                  f"(avg_score={avg_score:.3f}, std={std_score:.3f}, {reason})")

            self.threshold_history.append(self.confidence_threshold)

In [ ]:
# ТЕСТИРОВАНИЕ С АДАПТИВНОЙ TTA
import sys
sys.path.append('/content/TCT_data')
from tta_adaptive_tuned import AdaptiveTTA

# Загружаем модель
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_path = '/content/drive/MyDrive/TCT_models_simsiam/simsiam_fold1_BEST_map0.6718.pth'
checkpoint = torch.load(model_path, map_location='cpu')

model = create_model()
if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
else:
    model.load_state_dict(checkpoint)
model.to(device)
model.eval()

print(f" Модель загружена, устройство: {device}")

# Создаем адаптивную TTA
adaptive_tta = AdaptiveTTA(
    model=model,
    device=device
)

# Тестируем
test_dataset = TCTTestDataset('/content/TCT_data/data/fold', transforms=T.Compose([T.ToTensor()]))
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)

predictions = []

for images, img_paths, _ in tqdm(test_loader, desc="Адаптивная TTA"):
    image = images[0]
    result = adaptive_tta.predict_with_tta(image)
    predictions.append({
        'image_path': img_paths[0],
        'boxes': result['boxes'],
        'scores': result['scores']
    })

# Сохраняем результаты
adaptive_csv = '/content/TCT_data/adaptive_tuned_results.csv'
with open(adaptive_csv, 'w') as f:
    f.write('image_path,annotation\n')
    img_preds = {}
    total_preds = 0

    for pred in predictions:
        img_path = pred['image_path']
        if img_path not in img_preds:
            img_preds[img_path] = []

        for box, score in zip(pred['boxes'], pred['scores']):
            if score >= 0.5:
                x1, y1, x2, y2 = box
                pred_str = f"1 {score:.4f} {x1:.1f} {y1:.1f} {x2:.1f} {y2:.1f}"
                img_preds[img_path].append((score, pred_str))
                total_preds += 1

    for img_path, preds in img_preds.items():
        preds.sort(key=lambda x: x[0], reverse=True)
        pred_strings = [p[1] for p in preds[:100]]
        if pred_strings:
            annotation = ";".join(pred_strings)
            f.write(f'{img_path},{annotation}\n')
        else:
            f.write(f'{img_path},\n')

# Считаем mAP
from tool.voc_eval_new import custom_voc_eval

result = custom_voc_eval(
    gt_csv='/content/TCT_data/data/fold/test_for_map.csv',
    pred_csv=adaptive_csv,
    label_list=['1'],
    ovthresh=0.5,
    use_07_metric=False
)

adaptive_map = float(result[2]) if len(result) >= 3 else float(result[0])


print(f" Adaptive TTA mAP: {adaptive_map:.4f} ({adaptive_map*100:.2f}%)")
print(f" Всего предсказаний: {total_preds}")
print(f" Адаптаций: {adaptive_tta.adaptation_events}")
print(f" Финальный порог: {adaptive_tta.confidence_threshold:.2f}")

In [ ]:
# ФИНАЛЬНОЕ СРАВНЕНИЕ
print(f"1. Baseline (без TTA):         64.92%")
print(f"2. Обычная TTA:                 {best_tta_map*100:.2f}%")
print(f"3. Адаптивная TTA (настроена):  {adaptive_map*100:.2f}%")


# Определяем лучший метод
best_map = max(64.92, best_tta_map*100, adaptive_map*100)
best_methods = []
if best_map == 64.92:
    best_methods.append("Baseline")
if best_map == best_tta_map*100:
    best_methods.append("Обычная TTA")
if best_map == adaptive_map*100:
    best_methods.append("Адаптивная TTA")

print(f"\n ЛУЧШИЙ РЕЗУЛЬТАТ: {best_map:.2f}%")
print(f" Метод: {', '.join(best_methods)}")

# Сохраняем результат
with open('/content/drive/MyDrive/TCT_models_simsiam/final_best_result.txt', 'w') as f:
    f.write(f"Baseline (без TTA):        64.92%\n")
    f.write(f"Обычная TTA:                {best_tta_map*100:.2f}%\n")
    f.write(f"Адаптивная TTA (настроена): {adaptive_map*100:.2f}%\n")
    f.write(f"ЛУЧШИЙ РЕЗУЛЬТАТ: {best_map:.2f}%\n")
    f.write(f"МЕТОД: {', '.join(best_methods)}\n")

print("\nРезультаты сохранены в Google Drive")